# 02 - Data Cleaning

Compiled version of the cleaning notebook.

Design choices:
- Follow Shruti's audit-first cleaning logic.
- Avoid forward-filling by default because it can create artificial zero returns on cross-market holidays.
- Follow Wei-Han's return construction: log returns for price assets and level changes for VIX/yields.
- Treat Copper as a robustness extension, not part of the core modelling sample yet.
- Save core aligned prices/modelling variables plus a separate robustness variable file including Copper.


## Reader Orientation

This notebook protects the signal from data-quality mistakes. Since the dashboard will flag abnormal market behaviour, calendar mismatches or artificial forward-filled returns could create false warnings. The core output is a strictly aligned dataset for the main Brent-based nowcasting test.

In [21]:
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs" / "step02"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [22]:
CORE_ASSETS = ["Gold", "Brent", "DXY", "VIX", "US10Y"]
ROBUSTNESS_ASSETS = ["Copper"]
ALL_ASSETS = CORE_ASSETS + ROBUSTNESS_ASSETS

CORE_PRICE_ASSETS = ["Gold", "Brent", "DXY"]
ROBUSTNESS_PRICE_ASSETS = ["Copper"]
LEVEL_ASSETS = ["VIX", "US10Y"]

prices = pd.read_parquet(RAW_DIR / "prices_close_raw.parquet")
prices = prices[ALL_ASSETS].sort_index()

print("Raw close-price panel:", prices.shape)
prices.tail()


Raw close-price panel: (4134, 6)


Ticker,Gold,Brent,DXY,VIX,US10Y,Copper
Date,,,,,,
2026-05-27,4447.500000,94.290001,99.209999,16.290001,4.481,6.3050
2026-05-28,4499.299805,93.709999,99.019997,15.740000,4.455,6.3960
2026-05-29,4560.500000,92.050003,98.910004,15.320000,4.453,6.3595
2026-06-01,4475.200195,94.980003,99.199997,16.049999,4.475,6.5240
2026-06-02,4557.299805,93.669998,99.078003,16.160000,NaN,6.6480


In [23]:
missing_audit = pd.DataFrame({
    "missing_count": prices.isna().sum(),
    "missing_pct": prices.isna().mean().mul(100).round(3),
    "role": ["core" if asset in CORE_ASSETS else "robustness" for asset in prices.columns],
})

rows_with_missing = prices.loc[prices.isna().any(axis=1)].copy()

print("Rows with at least one missing value:", len(rows_with_missing))
missing_audit


Rows with at least one missing value: 43


,missing_count,missing_pct,role
Ticker,,,
Gold,7,0.169,core
Brent,37,0.895,core
DXY,5,0.121,core
VIX,5,0.121,core
US10Y,9,0.218,core
Copper,6,0.145,robustness


### Why Audit Missing Values First

The signal is based on abnormal returns and changing correlations, so data alignment errors can become false alarms. The latest missing-value audit shows all core assets have less than 1% missing data, with Brent the highest at 37 rows. That is small enough to use a strict aligned sample without materially damaging the dataset.

For the core modelling sample, we use strict alignment across Gold, Brent, DXY, VIX, and US10Y. Copper is cleaned separately for future robustness testing so that it does not change the core sample before the main signal is evaluated.

In [24]:
core_prices = prices[CORE_ASSETS].dropna().copy()

core_log_returns = np.log(core_prices[CORE_PRICE_ASSETS] / core_prices[CORE_PRICE_ASSETS].shift(1))
core_level_changes = core_prices[LEVEL_ASSETS].diff()

market_vars = pd.concat(
    [
        core_log_returns.rename(columns={asset: f"r_{asset}" for asset in CORE_PRICE_ASSETS}),
        core_level_changes.rename(columns={asset: f"d_{asset}" for asset in LEVEL_ASSETS}),
    ],
    axis=1,
).dropna()

print("Core aligned prices:", core_prices.shape)
print("Core market variables:", market_vars.shape)
market_vars.tail()


Core aligned prices: (4091, 5)
Core market variables: (4090, 5)


Ticker,r_Gold,r_Brent,r_DXY,d_VIX,d_US10Y
Date,,,,,
2026-05-26,-0.004567,-0.038997,-0.001511,0.309999,-0.065
2026-05-27,-0.011824,-0.054586,0.000403,-0.719999,-0.012
2026-05-28,0.011580,-0.006170,-0.001917,-0.550001,-0.026
2026-05-29,0.013510,-0.017873,-0.001111,-0.420000,-0.002
2026-06-01,-0.018881,0.031334,0.002928,0.730000,0.022


### Result Comment And Significance

The core modelling variables intentionally exclude Copper. This keeps the primary empirical test aligned with the original branch work: Gold as the alarm, Brent as the commodity book, and DXY/VIX/US10Y as interpretation variables. Log returns are used for Gold, Brent, and DXY because they are price-like assets; VIX and US10Y use level changes because they are already index/yield levels.

In [25]:
robustness_assets_available = [asset for asset in ROBUSTNESS_ASSETS if asset in prices.columns]
robustness_prices = prices[CORE_ASSETS + robustness_assets_available].dropna().copy()

robustness_price_assets = CORE_PRICE_ASSETS + robustness_assets_available
robustness_log_returns = np.log(robustness_prices[robustness_price_assets] / robustness_prices[robustness_price_assets].shift(1))
robustness_level_changes = robustness_prices[LEVEL_ASSETS].diff()

robustness_market_vars = pd.concat(
    [
        robustness_log_returns.rename(columns={asset: f"r_{asset}" for asset in robustness_price_assets}),
        robustness_level_changes.rename(columns={asset: f"d_{asset}" for asset in LEVEL_ASSETS}),
    ],
    axis=1,
).dropna()

print("Robustness aligned prices:", robustness_prices.shape)
print("Robustness market variables:", robustness_market_vars.shape)
robustness_market_vars.tail()


Robustness aligned prices: (4091, 6)
Robustness market variables: (4090, 6)


Ticker,r_Gold,r_Brent,r_DXY,r_Copper,d_VIX,d_US10Y
Date,,,,,,
2026-05-26,-0.004567,-0.038997,-0.001511,0.002991,0.309999,-0.065
2026-05-27,-0.011824,-0.054586,0.000403,-0.008843,-0.719999,-0.012
2026-05-28,0.011580,-0.006170,-0.001917,0.014330,-0.550001,-0.026
2026-05-29,0.013510,-0.017873,-0.001111,-0.005723,-0.420000,-0.002
2026-06-01,-0.018881,0.031334,0.002928,0.025538,0.730000,0.022


### Why Keep A Separate Robustness Sample

Copper is useful only after the core argument is established. Saving a separate robustness dataset lets us later ask whether the result survives a diversified Brent/Copper book without contaminating the first-pass signal design or changing the core sample length.

In [26]:
extreme_rows = []
for col in market_vars.columns:
    z = (market_vars[col] - market_vars[col].mean()) / market_vars[col].std()
    flagged = market_vars.loc[z.abs() > 5, [col]].copy()
    flagged["z_score"] = z.loc[flagged.index]
    flagged["variable"] = col
    flagged["sample"] = "core"
    extreme_rows.append(flagged.reset_index(names="date"))

for col in [c for c in robustness_market_vars.columns if c not in market_vars.columns]:
    z = (robustness_market_vars[col] - robustness_market_vars[col].mean()) / robustness_market_vars[col].std()
    flagged = robustness_market_vars.loc[z.abs() > 5, [col]].copy()
    flagged["z_score"] = z.loc[flagged.index]
    flagged["variable"] = col
    flagged["sample"] = "robustness"
    extreme_rows.append(flagged.reset_index(names="date"))

extreme_moves = pd.concat(extreme_rows, ignore_index=True) if extreme_rows else pd.DataFrame()
extreme_moves = extreme_moves.sort_values(["sample", "variable", "date"]) if not extreme_moves.empty else extreme_moves

print("Extreme move rows:", len(extreme_moves))
extreme_moves.head(30)


Extreme move rows: 49


Ticker,date,r_Gold,z_score,variable,sample,r_Brent,r_DXY,d_VIX,d_US10Y,r_Copper
45,2022-11-10,NaN,-5.974574,d_US10Y,core,NaN,NaN,NaN,-0.322,NaN
25,2010-05-10,NaN,-6.454927,d_VIX,core,NaN,NaN,-12.110001,NaN,NaN
26,2010-05-20,NaN,5.581737,d_VIX,core,NaN,NaN,10.470001,NaN,NaN
27,2011-08-08,NaN,8.529600,d_VIX,core,NaN,NaN,16.000000,NaN,NaN
28,2011-08-09,NaN,-6.897372,d_VIX,core,NaN,NaN,-12.939999,NaN,NaN
29,2011-08-18,NaN,5.912237,d_VIX,core,NaN,NaN,11.089998,NaN,NaN
30,2015-08-24,NaN,6.775808,d_VIX,core,NaN,NaN,12.710001,NaN,NaN
31,2018-02-05,NaN,10.667200,d_VIX,core,NaN,NaN,20.010000,NaN,NaN
32,2020-02-27,NaN,6.184103,d_VIX,core,NaN,NaN,11.600000,NaN,NaN
33,2020-03-09,NaN,6.674525,d_VIX,core,NaN,NaN,12.520000,NaN,NaN


### Result Comment And Significance

The extreme-move audit should not be treated as an automatic cleaning list. Large VIX, yield, Brent, or gold moves are often exactly the stress events the project wants to study. The purpose is to identify obvious data errors; if the extreme rows correspond to real market events, they should generally stay in the sample.

In [27]:
core_prices.to_parquet(PROCESSED_DIR / "prices_clean_core.parquet")
market_vars.to_parquet(PROCESSED_DIR / "market_vars_core.parquet")

# Compatibility aliases for later notebooks while the project is built step by step.
core_prices.to_parquet(PROCESSED_DIR / "prices_clean.parquet")
market_vars.to_parquet(PROCESSED_DIR / "market_vars.parquet")

robustness_prices.to_parquet(PROCESSED_DIR / "prices_clean_robustness.parquet")
robustness_market_vars.to_parquet(PROCESSED_DIR / "market_vars_robustness.parquet")

missing_audit.to_csv(OUTPUT_DIR / "missing_audit.csv")
rows_with_missing.to_csv(OUTPUT_DIR / "rows_with_missing.csv")
extreme_moves.to_csv(OUTPUT_DIR / "extreme_move_audit.csv", index=False)

print("Saved Step 02 core and robustness outputs.")


Saved Step 02 core and robustness outputs.
